---
last_verified: 2026-09-08
tool_version: n/a
---

# Exploring pyproject.toml tool tables

> L3 investigation: live-editing `[tool.ruff]`, `[tool.mypy]`, and `[tool.pytest.ini_options]` sections and observing how each tool reacts to config changes.

## Purpose

When multiple tools share a single `pyproject.toml`, a small config change in one `[tool.*]` section can unexpectedly affect another tool — or fail to take effect at all because the tool reads a different key path. This notebook walks through three common tool tables (`[tool.ruff]`, `[tool.mypy]`, `[tool.pytest.ini_options]`), edits each one live, and records the exact behavioral change. The goal is to build intuition for how pyproject.toml tool sections map to tool-specific configuration files.

## Step 1 — set up a minimal src-layout project

Create a tiny project with both a `src/` package and a `tests/` directory so every tool has something to inspect.

In [ ]:
import os, pathlib, textwrap

root = pathlib.Path('_explore_tool_tables')
src = root / 'src' / 'mylib'
tests = root / 'tests'
src.mkdir(parents=True, exist_ok=True)
tests.mkdir(parents=True, exist_ok=True)

(src / '__init__.py').write_text('')
(src / 'core.py').write_text(textwrap.dedent('''\
    from typing import Optional

    def greet(name: Optional[str] = None) -> str:
        if name is None:
            return "Hello, world!"
        return f"Hello, {name}!"

    unused_import = 42  # ruff F841 or E741 territory
'''))
(tests / 'test_core.py').write_text(textwrap.dedent('''\
    from mylib.core import greet

    def test_greet_default():
        assert greet() == "Hello, world!"

    def test_greet_named():
        assert greet("Alice") == "Hello, Alice!"
'''))

print('Project scaffolded at', root.resolve())

## Step 2 — start with a baseline pyproject.toml

Write a minimal config that satisfies all three tools at their defaults.

In [ ]:
baseline = textwrap.dedent('''\
            [build-system]
            requires = ["setuptools>=68.0"]
            build-backend = "setuptools.build_meta"

            [project]
            name = "explore-tool-tables"
            version = "0.1.0"
            requires-python = ">=3.10"
            dependencies = []

            [tool.setuptools.packages.find]
            where = ["src"]

            [tool.ruff]
            target-version = "py310"

            [tool.mypy]
            python_version = "3.10"

            [tool.pytest.ini_options]
            testpaths = ["tests"]
            addopts = "-v"
''')
(root / 'pyproject.toml').write_text(baseline)
print('Baseline pyproject.toml written')
print(baseline)

## Step 3 — ruff: adding lint rules and observing output

Start with no lint rules, then add `select` and `per-file-ignores` to see the difference.

In [ ]:
import subprocess

def run_ruff(*args):
    cmd = ['python', '-m', 'ruff', 'check', '--config', str(root / 'pyproject.toml'),
           str(root / 'src'), str(root / 'tests')] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f'$ ruff check {" ".join(args)}')
    print(result.stdout or '(clean)')
    if result.stderr:
        print('stderr:', result.stderr)
    return result

# Baseline — no rules selected, so ruff reports nothing
run_ruff()

In [ ]:
# Now add lint rules to pyproject.toml
ruff_config = textwrap.dedent('''\
            [build-system]
            requires = ["setuptools>=68.0"]
            build-backend = "setuptools.build_meta"

            [project]
            name = "explore-tool-tables"
            version = "0.1.0"
            requires-python = ">=3.10"
            dependencies = []

            [tool.setuptools.packages.find]
            where = ["src"]

            [tool.ruff]
            target-version = "py310"

            [tool.ruff.lint]
            select = ["E", "F", "I", "W"]

            [tool.mypy]
            python_version = "3.10"

            [tool.pytest.ini_options]
            testpaths = ["tests"]
            addopts = "-v"
''')
(root / 'pyproject.toml').write_text(ruff_config)

# Now ruff should flag unused imports, undefined names, etc.
run_ruff()

In [ ]:
# Add per-file-ignores to suppress warnings in tests
ruff_with_ignores = ruff_config.replace(
    'select = ["E", "F", "I", "W"]',
    'select = ["E", "F", "I", "W"]\n\n[tool.ruff.lint.per-file-ignores]\n"tests/**" = ["F841"]'
)
(root / 'pyproject.toml').write_text(ruff_with_ignores)

# The F841 in tests should now be suppressed
run_ruff()

### What changed

The `[tool.ruff.lint]` key (note: not just `[tool.ruff]`) is where rules live. Ruff ignores `[tool.ruff]` for rule selection — you must use `[tool.ruff.lint]` or it silently applies defaults. The `[tool.ruff.lint.per-file-ignores]` sub-table lets you carve out exceptions per directory, which is the standard pattern for allowing `assert` or unused variables in test files.

## Step 4 — mypy: strict mode and per-module overrides

Mypy reads `[tool.mypy]` at the top level. Nested `[tool.mypy.*]` sections are for per-module overrides, not sub-commands.

In [ ]:
def run_mypy():
    cmd = ['python', '-m', 'mypy', '--config-file', str(root / 'pyproject.toml'),
           str(root / 'src')]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print('$ mypy src/')
    print(result.stdout or '(clean)')
    if result.stderr:
        print('stderr:', result.stderr)
    return result

# Baseline — mypy with python_version = 3.10, no strict mode
run_mypy()

In [ ]:
# Enable strict mode — this activates dozens of additional checks
mypy_strict = ruff_with_ignores.replace(
    '[tool.mypy]\n            python_version = "3.10"',
    '[tool.mypy]\n            python_version = "3.10"\n            strict = true\n            warn_unused_configs = true'
)
(root / 'pyproject.toml').write_text(mypy_strict)
run_mypy()

In [ ]:
# Add a per-module override to relax mypy for tests
mypy_with_override = mypy_strict.replace(
    'warn_unused_configs = true',
    'warn_unused_configs = true\n\n[[tool.mypy.overrides]]\nmodule = "tests.*"\nignore_errors = true'
)
(root / 'pyproject.toml').write_text(mypy_with_override)
run_mypy()

### What changed

Key observations:
- `[tool.mypy]` (not `[tool.mypy.ini_options]`) is the correct section for mypy config in pyproject.toml.
- `strict = true` enables ~20 additional checks including `disallow_untyped_defs`, `no_implicit_optional`, and `warn_return_any`.
- `[[tool.mypy.overrides]]` is a TOML array of tables — each override is a separate `[[...]]` block, not a nested `[tool.mypy.overrides.module]`.
- Per-module overrides match on dotted module names: `tests.*` covers everything under the `tests/` package.

## Step 5 — pytest: ini_options and marker registration

Pytest reads `[tool.pytest.ini_options]` — note the `.ini_options` suffix. This is a historical artifact from when pytest config lived in `pytest.ini`.

In [ ]:
def run_pytest():
    cmd = ['python', '-m', 'pytest', '--co', '-q', '--rootdir', str(root)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print('$ pytest --co -q')
    print(result.stdout or '(no output)')
    if result.stderr:
        print('stderr:', result.stderr[:500])
    return result

# Baseline — just testpaths and addopts
run_pytest()

In [ ]:
# Add markers and change test discovery patterns
pytest_config = mypy_with_override.replace(
    'addopts = "-v"',
    'addopts = "-v --tb=short"\n            markers = ["slow: marks tests as slow (deselect with -m \'not slow\')"]\n            python_files = ["test_*.py"]\n            python_classes = ["Test*"]\n            python_functions = ["test_*"]'
)
(root / 'pyproject.toml').write_text(pytest_config)

# Now pytest knows about the 'slow' marker — no PytestUnknownMarkWarning
run_pytest()

### What changed

- `[tool.pytest.ini_options]` — the `.ini_options` suffix is mandatory; `[tool.pytest]` is silently ignored by pytest.
- `markers = [...]` registers custom markers so pytest doesn't warn about unknown markers on the command line.
- `python_files`, `python_classes`, `python_functions` control test discovery. The defaults are already `test_*.py`, `Test*`, and `test_*`, but writing them explicitly makes the config self-documenting.
- `addopts = "-v --tb=short"` passes default CLI flags to every pytest invocation — equivalent to adding `--tb=short` by hand each time.

## Step 6 — cross-tool interaction: line-length

Several tools care about line length. Ruff uses `[tool.ruff] line-length`, while mypy and pytest don't — but if you have a formatter that enforces 88 chars and a linter that warns on >79, you get conflicts.

In [ ]:
# Show the final pyproject.toml
final = (root / 'pyproject.toml').read_text()
print(final)

## Common errors and gotchas

| Mistake | Symptom | Fix |
|---|---|---|
| Using `[tool.ruff]` for rule selection | Rules are silently ignored | Use `[tool.ruff.lint]` |
| Using `[tool.pytest]` instead of `[tool.pytest.ini_options]` | Pytest finds no config | Use the full `.ini_options` suffix |
| Using `[tool.mypy.overrides]` as a sub-table | Override is silently ignored | Use `[[tool.mypy.overrides]]` (TOML array of tables) |
| Missing `markers = [...]` in pytest | `PytestUnknownMarkWarning` on `-m` usage | Register markers explicitly |
| Setting `line-length` in ruff but not the formatter | Lint passes but format fails | Set `[tool.ruff.format] line-length` too (or share via `[tool.ruff] line-length`) |

In [ ]:
# Clean up
import shutil
shutil.rmtree(root)
print('Cleaned up', root)

## Verify

The notebook creates a temporary project, edits pyproject.toml live, and runs each tool to confirm the behavioral change. Every config edit is followed by a tool invocation so the reader can see cause and effect in the same cell. The final cell cleans up the temporary directory.